# CrashDiag SFT — independent Kaggle notebook

This notebook starts from a fresh Kaggle GPU session. It clones CrashDiag at the dataset generator's exact source commit, reads only the `HF_TOKEN` Kaggle Secret, downloads and hash-verifies the four generated datasets from the private `devaanshpa/CrashDiag` Storage Bucket, trains the LoRA SFT adapter, and uploads every retained checkpoint plus the final adapter/state/metrics.

First run `python -m training.generate_dataset` on a trusted machine and copy its printed `RUN_ID` and `SOURCE_COMMIT` into the configuration cell below. Enable **GPU** and **Internet** in Kaggle before running all cells. The independent GRPO notebook reuses the same values. No Vultr sandbox credential is needed for SFT.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_ID = os.environ.get("CRASHDIAG_RUN_ID") or "PASTE_DATASET_RUN_ID_HERE"
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT") or "PASTE_DATASET_SOURCE_COMMIT_HERE"
EPOCHS = 3.0
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
SAVE_STEPS = 25
PRECISION = "auto"

if RUN_ID == "PASTE_DATASET_RUN_ID_HERE":
    raise ValueError("Set RUN_ID to the exact value printed by training.generate_dataset")
if SOURCE_COMMIT == "PASTE_DATASET_SOURCE_COMMIT_HERE":
    raise ValueError("Set SOURCE_COMMIT to the exact value printed by training.generate_dataset")
if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
    raise ValueError("SOURCE_COMMIT must be the full 40-character lowercase Git SHA printed by dataset generation")

print(f"RUN_ID={RUN_ID}")
print(f"SOURCE_COMMIT={SOURCE_COMMIT}")
print(f"bucket={BUCKET_ID}")
print(f"base_model={BASE_MODEL}")

## Install the exact repository revision

This cell is safe in a fresh session and fast-forwards an existing clean clone. It never writes credentials into the repository.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{REPO_BRANCH}"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-repository directory: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"Git checkout mismatch: expected {SOURCE_COMMIT}, got {CURRENT_COMMIT}")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_source_commit={CURRENT_COMMIT}")

## Load the Kaggle Secret and download the dataset stage

Create a Kaggle Secret named `HF_TOKEN` with fine-grained write access to `devaanshpa/CrashDiag`. The value is placed only in process memory and is never displayed or written to notebook output. The overall pipeline run is intentionally incomplete until GRPO finishes, but the downloaded dataset stage must have its own valid manifest and `_SUCCESS.json`.

In [ ]:
def required_secret(name: str) -> str:
    existing = os.environ.get(name)
    if existing:
        return existing
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach the Kaggle Secret {name!r} before continuing") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = RUN_ID

from training.artifacts import ArtifactConfig, ArtifactUploader

artifact_config = ArtifactConfig(
    bucket_id=BUCKET_ID,
    run_id=RUN_ID,
    prefix="runs",
    policy="required",
    local_root=REPO_DIR / "artifacts",
    token=os.environ["HF_TOKEN"],
)
uploader = ArtifactUploader(artifact_config)
RUN_DIR = artifact_config.local_run_root
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    uploader.verify_local_run(RUN_DIR, require_complete=False)
    print(f"verified existing dataset recovery directory: {RUN_DIR}")
else:
    uploader.download_run(RUN_DIR, allow_incomplete=True)
    print(f"downloaded pipeline artifacts: {uploader.remote_uri()}")

DATASET_DIR = RUN_DIR / "datasets"
required_datasets = [
    DATASET_DIR / "sft_train.jsonl",
    DATASET_DIR / "sft_eval.jsonl",
    DATASET_DIR / "grpo_train.jsonl",
    DATASET_DIR / "grpo_eval.jsonl",
    DATASET_DIR / "manifest.json",
    DATASET_DIR / "_SUCCESS.json",
]
missing = [str(path) for path in required_datasets if not path.is_file()]
if missing:
    raise RuntimeError(f"Refusing to train without completed bucket datasets: {missing}")
dataset_manifest = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
artifact_commit = dataset_manifest.get("runtime", {}).get("git_commit")
if artifact_commit != CURRENT_COMMIT:
    raise RuntimeError(f"Dataset/code revision mismatch: dataset={artifact_commit!r}, notebook={CURRENT_COMMIT!r}")
print(f"verified dataset stage: {uploader.remote_uri('datasets')}")

## Verify the Kaggle GPU

Automatic precision selects BF16 only when the device supports it and otherwise uses FP16.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is visible. Enable a GPU in Kaggle Notebook Settings.")
print(f"torch={torch.__version__}")
print(f"gpu={torch.cuda.get_device_name(0)}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")

## Inspect the verified datasets

The generator mechanically validates every SFT action before upload. This notebook additionally checks the downloaded row counts and confirms that GRPO rows contain no answer fields.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

sft_train_rows = read_jsonl(DATASET_DIR / "sft_train.jsonl")
sft_eval_rows = read_jsonl(DATASET_DIR / "sft_eval.jsonl")
grpo_train_rows = read_jsonl(DATASET_DIR / "grpo_train.jsonl")
grpo_eval_rows = read_jsonl(DATASET_DIR / "grpo_eval.jsonl")
if len(sft_train_rows) != len(grpo_train_rows) or len(sft_eval_rows) != len(grpo_eval_rows):
    raise RuntimeError("Downloaded SFT/GRPO dataset row counts diverge")
if not sft_train_rows or not sft_eval_rows:
    raise RuntimeError("Downloaded datasets are empty")
dataset_metadata = dataset_manifest.get("metadata", {})
if dataset_metadata.get("train_rows") != len(sft_train_rows) or dataset_metadata.get("eval_rows") != len(sft_eval_rows):
    raise RuntimeError("Downloaded row counts do not match the signed dataset manifest")
if not all(row.get("metadata", {}).get("mechanically_validated") is True for row in sft_train_rows + sft_eval_rows):
    raise RuntimeError("Downloaded SFT rows are not marked mechanically validated")
for row in grpo_train_rows + grpo_eval_rows:
    if any(field in row for field in ("completion", "answer", "expert_action")):
        raise RuntimeError("Downloaded GRPO data contains a target answer")
print(f"verified rows: train={len(sft_train_rows)}, eval={len(sft_eval_rows)}")

## Run LoRA SFT

The notebook uses the unit-tested training backend directly in this single-GPU process. Every save is synchronously copied to the private bucket before training proceeds.

In [ ]:
from training.sft import main as sft_main

sft_exit = sft_main([
    "--model", BASE_MODEL,
    "--dataset", str(DATASET_DIR / "sft_train.jsonl"),
    "--eval-dataset", str(DATASET_DIR / "sft_eval.jsonl"),
    "--output-dir", "outputs/sft",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--precision", PRECISION,
    "--save-strategy", "steps",
    "--save-steps", str(SAVE_STEPS),
    "--save-total-limit", "2",
    "--report-to", "none",
])
if sft_exit != 0:
    raise RuntimeError(f"SFT exited with {sft_exit}")

## Display the signed SFT training report

The trainer creates loss, learning-rate, and gradient-norm SVGs whenever those numeric series were logged. It also writes strict JSON history/summary files and a Markdown report before the SFT artifact manifest is finalized, so every displayed file is hash-covered in the private bucket. These are optimization diagnostics, not a success grader.

In [ ]:
from IPython.display import SVG, display

SFT_OUTPUT_DIR = REPO_DIR / "outputs/sft"
SFT_REPORT_DIR = SFT_OUTPUT_DIR / "reports"
SFT_SUMMARY_PATH = SFT_REPORT_DIR / "metrics_summary.json"
sft_charts = sorted(SFT_REPORT_DIR.glob("*.svg"))
if not SFT_SUMMARY_PATH.is_file() or not sft_charts:
    raise RuntimeError(f"SFT training report is incomplete: {SFT_REPORT_DIR}")
sft_summary = json.loads(SFT_SUMMARY_PATH.read_text(encoding="utf-8"))
print(json.dumps(sft_summary, indent=2, sort_keys=True))
for chart in sft_charts:
    print(f"displaying {chart.name}")
    display(SVG(filename=str(chart)))

sft_manifest_path = REPO_DIR / "artifacts" / RUN_ID / "sft/manifest.json"
sft_manifest = json.loads(sft_manifest_path.read_text(encoding="utf-8"))
manifest_paths = {entry["path"] for entry in sft_manifest.get("files", [])}
expected_reports = {path.relative_to(SFT_OUTPUT_DIR).as_posix() for path in SFT_REPORT_DIR.iterdir() if path.is_file()}
missing_reports = sorted(expected_reports - manifest_paths)
if missing_reports:
    raise RuntimeError(f"SFT report files are absent from the signed bucket manifest: {missing_reports}")
print(f"signed SFT reports uploaded: {uploader.remote_uri('sft')}/reports")

## Verify the handoff to GRPO

Do not mark the whole run complete yet; GRPO and mechanical evaluation are separate stages. Copy the printed `RUN_ID` and `SOURCE_COMMIT` into the GRPO notebook.

In [ ]:
required_outputs = [
    REPO_DIR / "outputs/sft/adapter_config.json",
    REPO_DIR / "artifacts" / RUN_ID / "datasets/_SUCCESS.json",
    REPO_DIR / "artifacts" / RUN_ID / "sft/manifest.json",
    REPO_DIR / "artifacts" / RUN_ID / "sft/_SUCCESS.json",
    REPO_DIR / "outputs/sft/reports/metrics_summary.json",
]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise RuntimeError(f"SFT handoff is incomplete: {missing}")
Path("/kaggle/working/crashdiag_handoff.txt").write_text(
    f"RUN_ID={RUN_ID}\nSOURCE_COMMIT={SOURCE_COMMIT}\n", encoding="utf-8"
)
print("SFT stage complete and uploaded.")
print(f"RUN_ID={RUN_ID}")
print(f"SOURCE_COMMIT={SOURCE_COMMIT}")
print(f"GRPO input: {uploader.remote_uri('sft')}")